# 1. 환경변수 설정

In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv(Path(".env"), override=True)
embedding_model = OpenAIEmbeddings(model=os.getenv("EMBEDED_MODEL"))

# 2. 새 DB(chroma4) 적재
청크 앞에 `[제목 | 캐릭터]` 접두어를 붙이고, 에피소드마다 objects 청크를 추가

In [2]:
import csv
import json
import re
from pathlib import Path
from langchain_chroma import Chroma
from langchain_core.documents import Document

RAW_DIR = Path("raw_json")
CSV_PATH = "episodes.csv"
NEW_CHROMA_DIR = "chroma4"

# episodes.csv에서 제목, 링크 가져오기
with open(CSV_PATH, encoding="utf-8-sig") as f:
    episode_info = {row["episode_id"]: row for row in csv.DictReader(f)}


def short_title(title):
    """'[전편공개]먼작귀(먼가 작고 귀여운 녀석) 1화_단단한 푸딩, 핫케이크' -> '1화 단단한 푸딩, 핫케이크'"""
    m = re.search(r"(\d+화)_?(.*)", title)
    return f"{m.group(1)} {m.group(2)}".strip() if m else title


def to_documents_v2(episode_id, raw):
    data = json.loads(raw)
    info = episode_info[episode_id]
    title = short_title(info["title"])
    characters = ", ".join(dict.fromkeys(data.get("characters", [])))
    objects = ", ".join(data.get("objects", []))

    # 모든 청크 앞에 붙는 접두어: 제목(음식 이름이 많이 들어 있음) + 등장 캐릭터
    prefix = f"[{title} | {characters}]"
    base_meta = {
        "episode_id": episode_id,
        "title": title,
        "video_url": info["video_url"],
        "characters": characters,
        "objects": objects,
    }

    docs, ids = [], []

    def add(field, text, doc_id):
        docs.append(Document(
            page_content=f"{prefix}\n{text}",
            metadata={**base_meta, "field": field},
        ))
        ids.append(doc_id)

    add("summary", data["summary"], f"{episode_id}-summary")
    for i, scene in enumerate(data.get("key_scenes", [])):
        add("key_scene", scene, f"{episode_id}-scene-{i}")
    if objects:
        add("objects", f"등장하는 음식·물건·장소: {objects}", f"{episode_id}-objects")

    return docs, ids


vectorstore = Chroma(
    collection_name="chiikawa",
    embedding_function=embedding_model,
    persist_directory=NEW_CHROMA_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)

all_docs, all_ids = [], []
for path in sorted(RAW_DIR.glob("*.json")):
    docs, ids = to_documents_v2(path.stem, path.read_text(encoding="utf-8"))
    all_docs += docs
    all_ids += ids

# 이미 적재돼 있으면 다시 임베딩하지 않음 (재실행 시 비용 절약)
if vectorstore._collection.count() < len(all_ids):
    vectorstore.add_documents(all_docs, ids=all_ids)
print("적재 완료:", vectorstore._collection.count(), "건")
print(vectorstore.get(ids=["ep005-objects"])["documents"][0])

적재 완료: 1304 건
[5화 까리메라, 까르메라 | 치이카와, 하치와레, 우사기]
등장하는 음식·물건·장소: 크레용, 종이, 까르메라우 라면, 냄비, 그릇, 젓가락, 꽃 화관, 초원


# 3. 검색 비교

In [3]:
questions = [
    "치이카와가 라면 먹는 에피소드 알려줘",
    "치이카와가 피자 만두 먹는 장면",
    "푸딩 나오는 에피소드",
    "치이카와 작가가 누구야?",
    "오늘 서울 날씨 어때?",
]

for q in questions:
    print(q)
    for d, score in vectorstore.similarity_search_with_relevance_scores(q, k=3):
        first_line, _, body = d.page_content.partition("\n")
        print(f"  {score:.3f}  {d.id}  {first_line}  {body[:30]}")
    print()

치이카와가 라면 먹는 에피소드 알려줘


  0.555  ep075-scene-1  [75화 다 먹을 수 없어 | 치이카와, 하치와레, 우사기]  치이카와와 하치와레가 거대한 오므라이스 위에 올라가 맛
  0.554  ep241-scene-0  [241화 고추냉이 맛 과자 | 치이카와, 하치와레, 쿠리만주]  하치와레가 병에 든 소다맛 음료를 컵에 따라 마시는 장
  0.549  ep242-scene-2  [242화 어묵탕 국물 | 치이카와, 하치와레, 쿠리만주]  치이카와와 하치와레가 과일 탄산음료를 들고 오뎅과 무를

치이카와가 피자 만두 먹는 장면
  0.674  ep126-scene-2  [126화 과자를 샀어 | 치이카와, 기타]  치이카와와 다른 캐릭터가 나란히 앉아 과자를 나누어 먹
  0.672  ep126-scene-1  [126화 과자를 샀어 | 치이카와, 기타]  치이카와가 개구리 모양의 자판기에 동전을 넣고 과자가 
  0.661  ep241-scene-2  [241화 고추냉이 맛 과자 | 치이카와, 하치와레, 쿠리만주]  치이카와와 하치와레가 쿠리만주와 함께 건배를 하며 기뻐

푸딩 나오는 에피소드


  0.453  ep111-objects  [111화 엄청 맛있는 푸딩, 응모 | 쿠리만주, 하치와레, 우사기, 치이카와]  등장하는 음식·물건·장소: 푸딩, 숟가락, 전단지, 가
  0.440  ep111-scene-1  [111화 엄청 맛있는 푸딩, 응모 | 쿠리만주, 하치와레, 우사기, 치이카와]  쿠리만주, 하치와레, 우사기가 모여 앉아 맛있게 푸딩을
  0.435  ep001-objects  [1화 단단한 푸딩, 핫케이크 | 치이카와, 하치와레, 우사기]  등장하는 음식·물건·장소: 푸딩, 핫케이크, 뒤집개, 

치이카와 작가가 누구야?
  0.497  ep018-summary  [18화 풀뽑기, 커플 무기 | 치이카와, 하치와레, 우사기, 쿠리만주, 기타]  치이카와는 일자리를 구하기 위해 노동 장소로 향하지만,
  0.494  ep229-summary  [229화 검은 별똥별 후편8 | 치이카와, 기타]  치이카와는 친구들이 인터넷으로 무언가를 열심히 검색하는
  0.488  ep100-scene-0  [100화 정말로 있었던 무서운 이야기 5 | 치이카와, 하치와레, 우사기]  치이카와와 하치와레가 겁에 질린 표정으로 창밖의 그림자

오늘 서울 날씨 어때?


  0.330  ep157-objects  [157화 날 봐, 츳츳츳 | 모몽가, 기타]  등장하는 음식·물건·장소: 꽃, 그루터기, 숲
  0.310  ep197-scene-2  [197화 오데와 함께 감옥 9 | 기타]  하늘다람쥐가 꽃이 꽂힌 콩국수 그릇을 들고 당당한 표정
  0.301  ep196-scene-3  [196화 오데와 함께 감옥 8 | 하치와레, 치이카와, 우사기, 모몽가, 기타]  외눈박이 거인이 가져다준 콩스프 그릇을 들고 마시는 하



# 4. 하이브리드 검색 (BM25 + 벡터)
- BM25: 단어가 실제로 들어 있는지 보는 키워드 검색. 한국어 조사 처리를 위해 kiwipiepy로 명사만 뽑음
- 벡터: 기존 의미 검색
- 두 순위를 RRF로 합침: 양쪽에서 모두 상위인 문서가 위로 올라옴

In [4]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

kiwi = Kiwi()
KEEP_TAGS = ("NNG", "NNP", "SL", "SN", "XR")   # 명사, 외국어, 숫자, 어근만 사용 (조사·동사 제거)
# 거의 모든 질문·문서에 나와서 검색을 흐리는 단어
STOPWORDS = {"치이카와", "먼작귀", "에피소드", "장면", "내용", "애니"}


def tokenize(text):
    """'치이카와가 라면 먹는 장면' -> ['라면']"""
    return [t.form for t in kiwi.tokenize(text)
            if t.tag in KEEP_TAGS and t.form not in STOPWORDS]


# chroma4에 저장된 문서 전체로 BM25 인덱스 생성
stored = vectorstore.get(include=["documents", "metadatas"])
bm25_docs = [
    Document(id=i, page_content=text, metadata=meta)
    for i, text, meta in zip(stored["ids"], stored["documents"], stored["metadatas"])
]
bm25 = BM25Okapi([tokenize(d.page_content) for d in bm25_docs])
print("BM25 인덱스:", len(bm25_docs), "건")
print(tokenize("치이카와가 피자 만두 먹는 장면"))

BM25 인덱스: 1304 건
['피자', '만두']


In [5]:
RRF_K = 60   # RRF 상수 (보통 60)


def search_bm25(question, k=10):
    scores = bm25.get_scores(tokenize(question))
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [bm25_docs[i] for i in top if scores[i] > 0]


def search_vector(question, k=10):
    return [d for d, _ in vectorstore.similarity_search_with_relevance_scores(question, k=k)]


def hybrid_search(question, k=3, fetch_k=10):
    """BM25 순위와 벡터 순위를 RRF(Reciprocal Rank Fusion)로 합친다."""
    fused = {}  # doc_id -> {"doc", "score", "bm25", "vector"}
    for name, docs in (("bm25", search_bm25(question, fetch_k)),
                       ("vector", search_vector(question, fetch_k))):
        for rank, d in enumerate(docs, start=1):
            item = fused.setdefault(d.id, {"doc": d, "score": 0.0, "bm25": None, "vector": None})
            item["score"] += 1 / (RRF_K + rank)
            item[name] = rank
    return sorted(fused.values(), key=lambda x: x["score"], reverse=True)[:k]

# 5. 하이브리드 검색 비교

In [6]:
questions = [
    "치이카와가 라면 먹는 에피소드 알려줘",
    "치이카와가 피자 만두 먹는 장면",
    "푸딩 나오는 에피소드",
    "치이카와 작가가 누구야?",
    "오늘 서울 날씨 어때?",
]

for q in questions:
    print(q, "->", tokenize(q))
    for item in hybrid_search(q, k=3):
        d = item["doc"]
        first_line, _, body = d.page_content.partition("\n")
        print(f"  {item['score']:.4f}  (BM25 {item['bm25']}위 / 벡터 {item['vector']}위)  "
              f"{d.id}  {first_line}  {body[:30]}")
    print()

치이카와가 라면 먹는 에피소드 알려줘 -> ['라면']
  0.0164  (BM25 1위 / 벡터 None위)  ep005-scene-2  [5화 까리메라, 까르메라 | 치이카와, 하치와레, 우사기]  치이카와와 하치와레가 마주 앉아 맛있게 라면을 후루룩 
  0.0164  (BM25 None위 / 벡터 1위)  ep075-scene-1  [75화 다 먹을 수 없어 | 치이카와, 하치와레, 우사기]  치이카와와 하치와레가 거대한 오므라이스 위에 올라가 맛
  0.0161  (BM25 2위 / 벡터 None위)  ep005-summary  [5화 까리메라, 까르메라 | 치이카와, 하치와레, 우사기]  치이카와가 방 안에서 그림을 그리는 동안, 하치와레가 

치이카와가 피자 만두 먹는 장면 -> ['피자', '만두']


  0.0294  (BM25 8위 / 벡터 8위)  ep115-scene-1  [115화 피자 도우 | 치이카와, 하치와레, 우사기, 기타]  우사기가 피자를 잡기 위해 높게 점프하지만 손이 닿지 
  0.0164  (BM25 1위 / 벡터 None위)  ep004-scene-2  [4화 브로콜리, 피자만두 | 치이카와, 우사기]  우사기가 거대한 피자 만두를 한 입 베어 물고 치즈가 
  0.0164  (BM25 None위 / 벡터 1위)  ep126-scene-2  [126화 과자를 샀어 | 치이카와, 기타]  치이카와와 다른 캐릭터가 나란히 앉아 과자를 나누어 먹

푸딩 나오는 에피소드 -> ['푸딩']
  0.0320  (BM25 3위 / 벡터 2위)  ep111-scene-1  [111화 엄청 맛있는 푸딩, 응모 | 쿠리만주, 하치와레, 우사기, 치이카와]  쿠리만주, 하치와레, 우사기가 모여 앉아 맛있게 푸딩을
  0.0311  (BM25 8위 / 벡터 1위)  ep111-objects  [111화 엄청 맛있는 푸딩, 응모 | 쿠리만주, 하치와레, 우사기, 치이카와]  등장하는 음식·물건·장소: 푸딩, 숟가락, 전단지, 가
  0.0310  (BM25 6위 / 벡터 3위)  ep001-objects  [1화 단단한 푸딩, 핫케이크 | 치이카와, 하치와레, 우사기]  등장하는 음식·물건·장소: 푸딩, 핫케이크, 뒤집개, 

치이카와 작가가 누구야? -> ['작가']


  0.0164  (BM25 None위 / 벡터 1위)  ep018-summary  [18화 풀뽑기, 커플 무기 | 치이카와, 하치와레, 우사기, 쿠리만주, 기타]  치이카와는 일자리를 구하기 위해 노동 장소로 향하지만,
  0.0161  (BM25 None위 / 벡터 2위)  ep229-summary  [229화 검은 별똥별 후편8 | 치이카와, 기타]  치이카와는 친구들이 인터넷으로 무언가를 열심히 검색하는
  0.0159  (BM25 None위 / 벡터 3위)  ep100-scene-0  [100화 정말로 있었던 무서운 이야기 5 | 치이카와, 하치와레, 우사기]  치이카와와 하치와레가 겁에 질린 표정으로 창밖의 그림자

오늘 서울 날씨 어때? -> ['서울', '날씨']
  0.0164  (BM25 1위 / 벡터 None위)  ep106-scene-1  [106화 포쉐트 | 치이카와, 하치와레, 우사기]  천둥 번개가 치고 비가 내리는 날씨 속에서 치이카와와 
  0.0164  (BM25 None위 / 벡터 1위)  ep157-objects  [157화 날 봐, 츳츳츳 | 모몽가, 기타]  등장하는 음식·물건·장소: 꽃, 그루터기, 숲
  0.0161  (BM25 2위 / 벡터 None위)  ep105-scene-0  [105화 팬레터, 닭튀김 레몬 | 치이카와, 하치와레, 우사기, 기타]  갑옷 씨가 노점에 앉아 쌀쌀한 날씨에 웅크리고 있는 장

